In [ ]:
# Install Ultralytics library for YOLOv8 training
!pip install -q ultralytics
!pip install roboflow

from roboflow import Roboflow
import os
import shutil
import glob
import yaml
import torch
from google.colab import drive
import random
import xml.etree.ElementTree as ET
from collections import defaultdict
from pathlib import Path
from ultralytics import YOLO

print("Environment ready and libraries imported!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 10.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Environment ready and libraries imported!


In [ ]:
#GPU check
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# import torch
# print(torch.cuda.is_available())        # Should print: True
# print(torch.version.cuda)               # Should print: 11.8
# print(torch.cuda.get_device_name(0))    # Should display your workstation's GPU model


In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
rf = Roboflow(api_key="6VXvP1VLsNubz6wiRVLi")
project = rf.workspace("syed-fathaullah").project("sponsor-detector")
version = project.version(2)
dataset = version.download("yolov8")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.3
    Uninstalling typer-0.27.3:
      Successfully uninstalled typer-0.27.3
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Sponsor-detector-2 in yolov8:: 100%|██████████| 5350/5350 [00:00<00:00, 6734.55it/s]


### Step 10: Fine-tune/Train on a New Dataset using Saved Weights
Run this cell to load your previously trained custom weights (`best.pt`) and continue training on a new dataset config.

In [ ]:
# 1. Path to your previously trained custom weights
custom_weights_path = '/content/drive/MyDrive/Colab Notebooks/CSC490/logodet/best.pt'

# 2. Path to your NEW dataset YAML configuration file
# Replace this with the actual path to your new dataset's YAML file
new_dataset_yaml = '/content/Sponsor-detector-2/data.yaml'

# Check if a GPU is available, otherwise use CPU
device_to_use = 0 if torch.cuda.is_available() else 'cpu'

if os.path.exists(custom_weights_path):
    print(f"Loading existing custom weights from: {custom_weights_path}")
    # Initialize the model with your custom weights instead of 'yolov8n.pt'
    fine_tune_model = YOLO(custom_weights_path)

    # 3. Train on the new dataset
    print(f"Starting training on device: {device_to_use}")
    results = fine_tune_model.train(
        data=new_dataset_yaml,
        epochs=20,          # Set your desired epochs for fine-tuning
        imgsz=640,
        batch=16,
        device=device_to_use, # Use GPU if available, else CPU
        project='logo_detection',
        name='yolov8_logos_finetuned'
    )
    print("Fine-tuning process finished!")
else:
    print(f"[ERROR] Custom weights not found at: {custom_weights_path}. Please verify the path or complete a previous training run first.")

Loading existing custom weights from: /content/drive/MyDrive/Colab Notebooks/CSC490/logodet/best.pt
Starting training on device: 0
Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Sponsor-detector-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, 

In [ ]:
import shutil
from google.colab import files

# Define the paths
run_dir = '/content/runs/detect/logo_detection/yolov8_logos_finetuned'
zip_name = '/content/yolov8_logos_finetuned_results'

# Compress the results folder into a .zip file
print("Zipping the results folder...")
shutil.make_archive(zip_name, 'zip', run_dir)

# Download the zip file to your local computer
print("Downloading the zip file...")
files.download(f"{zip_name}.zip")

Zipping the results folder...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>